In [11]:
import pandas as pd

In [12]:
df = pd.read_excel(r"C:\Order Orchestration code\order_orchestration\data\outputs_30oct26\unified_execution_decisions.xlsx",engine='openpyxl')
df.head()

,order_id,order_category,colour_code,order_create_datetime,decision_timestamp,material_code,dealer_code,dealer_norms,parent_plant,decision,...,source_sla_days,source_rlt_days,transport_cost,expected_dispatch,expected_delivery,dispatch_datetime,inbound_rlt_days,expected_arrival_datetime,arrival_days_from_decision,reason_codes
0,ROW_317,ZMSO,16,2026-10-30 03:01:07,2026-10-30 10:28:33.358,IZ300499,1S2169,150.0,1159,CREDIT_BLOCKED,...,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaT,NaN,"CREDIT_BLOCKED,REMAINING_QTY_HELD_AT_PARENT"
1,ROW_323,ZMSO,16,2026-10-30 03:01:07,2026-10-30 10:28:33.358,ID630893,1S2169,24.0,1159,CREDIT_BLOCKED,...,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaT,NaN,"CREDIT_BLOCKED,REMAINING_QTY_HELD_AT_PARENT"
2,ROW_332,ZMSO,16,2026-10-30 03:01:07,2026-10-30 10:28:33.358,ID630893,1S7178,24.0,1158,CREDIT_BLOCKED,...,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaT,NaN,"CREDIT_BLOCKED,REMAINING_QTY_HELD_AT_PARENT"
3,ROW_333,ZMSO,16,2026-10-30 03:01:07,2026-10-30 10:28:33.358,ID630893,1S2313,24.0,1152,CREDIT_BLOCKED,...,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaT,NaN,"CREDIT_BLOCKED,REMAINING_QTY_HELD_AT_PARENT"
4,ROW_336,ZMSO,16,2026-10-30 03:01:07,2026-10-30 10:28:33.358,ID630893,1S2060,24.0,1152,CREDIT_BLOCKED,...,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaT,NaN,"CREDIT_BLOCKED,REMAINING_QTY_HELD_AT_PARENT"


In [13]:
df.columns

Index(['order_id', 'order_category', 'colour_code', 'order_create_datetime',
       'decision_timestamp', 'material_code', 'dealer_code', 'dealer_norms',
       'parent_plant', 'decision', 'selected_source', 'selected_source_type',
       'allocation_sequence', 'original_requested_quantity',
       'effective_requested_quantity', 'moq_qty', 'source_allocated_quantity',
       'source_git_reserved_quantity', 'total_physical_allocated_quantity',
       'total_git_reserved_quantity', 'total_fulfilled_quantity',
       'remaining_quantity', 'remaining_quantity_owner', 'transport_mode',
       'source_sla_days', 'source_rlt_days', 'transport_cost',
       'expected_dispatch', 'expected_delivery', 'dispatch_datetime',
       'inbound_rlt_days', 'expected_arrival_datetime',
       'arrival_days_from_decision', 'reason_codes'],
      dtype='str')

In [14]:
import numpy as np

In [15]:
import pandas as pd


# =========================================================
# 1. PREPARE DECISION DATAFRAME
# =========================================================
sap_df = df.copy()


# ---------------------------------------------------------
# Validate required columns
# ---------------------------------------------------------
required_columns = [
    "order_id",
    "order_category",
    "colour_code",
    "order_create_datetime",
    "material_code",
    "dealer_code",
    "parent_plant",
    "decision",
    "selected_source",
    "source_allocated_quantity",
    "source_git_reserved_quantity",
    "remaining_quantity",
]

missing_columns = [
    column
    for column in required_columns
    if column not in sap_df.columns
]

if missing_columns:
    raise ValueError(
        "Decision dataframe is missing required columns: "
        f"{missing_columns}. "
        f"Available columns: {sap_df.columns.tolist()}"
    )


# ---------------------------------------------------------
# Normalize text columns
# ---------------------------------------------------------
for column in [
    "order_id",
    "order_category",
    "decision",
    "material_code",
    "dealer_code",
    "parent_plant",
    "selected_source",
]:
    sap_df[column] = (
        sap_df[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )


# ---------------------------------------------------------
# Remove trailing .0 from code fields
# ---------------------------------------------------------
for column in [
    "material_code",
    "dealer_code",
    "parent_plant",
    "selected_source",
]:
    sap_df[column] = (
        sap_df[column]
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
    )


# ---------------------------------------------------------
# Normalize original colour code
# ---------------------------------------------------------
sap_df["colour_code"] = pd.to_numeric(
    sap_df["colour_code"],
    errors="coerce",
).astype("Int64")


# ---------------------------------------------------------
# Normalize quantity columns
# ---------------------------------------------------------
quantity_columns = [
    "source_allocated_quantity",
    "source_git_reserved_quantity",
    "remaining_quantity",
]

for column in quantity_columns:
    sap_df[column] = pd.to_numeric(
        sap_df[column],
        errors="coerce",
    ).fillna(0)


# =========================================================
# 2. VALIDATE ORDER_ID
# =========================================================
# Since line_id is not present, order_id must identify one
# material-dealer-colour-parent combination.
# =========================================================
order_id_validation = (
    sap_df.groupby(
        "order_id",
        dropna=False,
    )
    .agg(
        material_count=(
            "material_code",
            "nunique",
        ),
        dealer_count=(
            "dealer_code",
            "nunique",
        ),
        colour_count=(
            "colour_code",
            "nunique",
        ),
        parent_count=(
            "parent_plant",
            "nunique",
        ),
    )
)

invalid_order_ids = order_id_validation[
    (
        order_id_validation[
            "material_count"
        ] > 1
    )
    | (
        order_id_validation[
            "dealer_count"
        ] > 1
    )
    | (
        order_id_validation[
            "colour_count"
        ] > 1
    )
    | (
        order_id_validation[
            "parent_count"
        ] > 1
    )
]

if not invalid_order_ids.empty:
    raise ValueError(
        "Some order_id values represent multiple order lines. "
        "SAP output cannot safely group these rows by order_id.\n"
        f"{invalid_order_ids.head(20).to_string()}"
    )


# =========================================================
# 3. PREPARE DATE AND TIME
# =========================================================
order_datetime = pd.to_datetime(
    sap_df["order_create_datetime"],
    errors="coerce",
)

sap_df["DATE"] = (
    order_datetime.dt.strftime("%Y%m%d")
)

sap_df["TIME"] = (
    order_datetime.dt.strftime("%H%M%S")
)


# =========================================================
# 4. HELPER FUNCTIONS
# =========================================================
def get_allocation_colour_code(
    original_colour,
) -> int | None:
    """
    SAP colour for physical or GIT allocation.

    Original 16 -> SAP 1
    Original 17 -> SAP 2
    """

    if pd.isna(original_colour):
        return None

    colour = int(original_colour)

    if colour == 16:
        return 1

    if colour == 17:
        return 2

    return None


def get_remaining_colour_code(
    original_colour,
    total_allocated_quantity,
) -> int | None:
    """
    SAP colour for remaining quantity.

    Original colour 16:
        No allocation      -> SAP colour 6
        Partial allocation -> SAP colour 7

    Original colour 17:
        No allocation      -> SAP colour 7
        Partial allocation -> SAP colour 8
    """

    if pd.isna(original_colour):
        return None

    colour = int(original_colour)

    allocated_quantity = pd.to_numeric(
        total_allocated_quantity,
        errors="coerce",
    )

    if pd.isna(allocated_quantity):
        allocated_quantity = 0

    partial_allocation_happened = (
        allocated_quantity > 0
    )

    if colour == 16:
        if partial_allocation_happened:
            return 7

        return 6

    if colour == 17:
        if partial_allocation_happened:
            return 8

        return 7

    return None


def clean_code(
    value,
) -> str | None:
    """
    Normalizes plant, dealer, and material codes.
    """

    if pd.isna(value):
        return None

    cleaned_value = (
        str(value)
        .strip()
        .upper()
    )

    if cleaned_value.endswith(".0"):
        cleaned_value = cleaned_value[:-2]

    if cleaned_value in {
        "",
        "NAN",
        "NONE",
        "NULL",
        "<NA>",
        "NAT",
    }:
        return None

    return cleaned_value


def format_sap_quantity(
    value,
):
    """
    Returns an integer for whole-number quantities.
    Otherwise returns the decimal value.
    """

    numeric_value = pd.to_numeric(
        value,
        errors="coerce",
    )

    if pd.isna(numeric_value):
        return 0

    numeric_value = float(
        numeric_value
    )

    if numeric_value.is_integer():
        return int(
            numeric_value
        )

    return numeric_value


# =========================================================
# 5. CREATE PHYSICAL AND GIT ALLOCATION ROWS
# =========================================================
sap_output_rows: list[dict] = []


for _, row in sap_df.iterrows():
    decision = str(
        row["decision"]
    ).strip().upper()

    allocation_colour = (
        get_allocation_colour_code(
            row["colour_code"]
        )
    )

    # Only process supported colours 16 and 17.
    if allocation_colour is None:
        continue

    # -----------------------------------------------------
    # Physical allocation row
    # -----------------------------------------------------
    physical_quantity = pd.to_numeric(
        row["source_allocated_quantity"],
        errors="coerce",
    )

    if (
        pd.notna(physical_quantity)
        and physical_quantity > 0
    ):
        sap_output_rows.append(
            {
                # Internal key. Removed from final SAP output.
                "order_id": row["order_id"],

                "From Location": clean_code(
                    row["selected_source"]
                ),

                "Material Code": clean_code(
                    row["material_code"]
                ),

                "To Location": clean_code(
                    row["dealer_code"]
                ),

                "Quantity": format_sap_quantity(
                    physical_quantity
                ),

                # Original 16 -> 1
                # Original 17 -> 2
                "Colour Code": allocation_colour,

                "Order Type": (
                    row["order_category"]
                ),

                "DATE": row["DATE"],

                "TIME": row["TIME"],

                "GIT Flag": 0,

                # Physical allocation appears first.
                "_row_type_order": 1,
            }
        )

    # -----------------------------------------------------
    # GIT reservation row
    # -----------------------------------------------------
    git_quantity = pd.to_numeric(
        row["source_git_reserved_quantity"],
        errors="coerce",
    )

    if (
        decision
        == "DEFERRED_GIT_ALLOCATION"
        and pd.notna(git_quantity)
        and git_quantity > 0
    ):
        sap_output_rows.append(
            {
                # Internal key. Removed from final SAP output.
                "order_id": row["order_id"],

                # GIT allocation is executed through the
                # order's parent plant.
                "From Location": clean_code(
                    row["parent_plant"]
                ),

                "Material Code": clean_code(
                    row["material_code"]
                ),

                "To Location": clean_code(
                    row["dealer_code"]
                ),

                "Quantity": format_sap_quantity(
                    git_quantity
                ),

                # GIT is treated as allocated quantity.
                # Original 16 -> 1
                # Original 17 -> 2
                "Colour Code": allocation_colour,

                "Order Type": (
                    row["order_category"]
                ),

                "DATE": row["DATE"],

                "TIME": row["TIME"],

                "GIT Flag": 1,

                # GIT appears after physical allocation.
                "_row_type_order": 2,
            }
        )


# =========================================================
# 6. CREATE ORDER-LEVEL SUMMARY
# =========================================================
# The decision dataframe can contain multiple source rows for
# one order. Aggregate physical and GIT quantities by order_id.
#
# remaining_quantity is repeated on each source row, so use
# min instead of sum.
# =========================================================
sort_columns = [
    "order_id",
]

if "allocation_sequence" in sap_df.columns:
    sort_columns.append(
        "allocation_sequence"
    )


order_level_rows = (
    sap_df.sort_values(
        by=sort_columns,
        na_position="last",
    )
    .groupby(
        "order_id",
        as_index=False,
        dropna=False,
    )
    .agg(
        order_category=(
            "order_category",
            "first",
        ),

        colour_code=(
            "colour_code",
            "first",
        ),

        material_code=(
            "material_code",
            "first",
        ),

        dealer_code=(
            "dealer_code",
            "first",
        ),

        parent_plant=(
            "parent_plant",
            "first",
        ),

        DATE=(
            "DATE",
            "first",
        ),

        TIME=(
            "TIME",
            "first",
        ),

        # Sum all physical-source quantities.
        physical_allocated_quantity=(
            "source_allocated_quantity",
            "sum",
        ),

        # Sum all GIT reservation quantities.
        git_reserved_quantity=(
            "source_git_reserved_quantity",
            "sum",
        ),

        # Repeated on each decision row.
        remaining_quantity=(
            "remaining_quantity",
            "min",
        ),
    )
)


# ---------------------------------------------------------
# Calculate total allocated quantity per order
# ---------------------------------------------------------
order_level_rows[
    "total_allocated_quantity"
] = (
    order_level_rows[
        "physical_allocated_quantity"
    ]
    + order_level_rows[
        "git_reserved_quantity"
    ]
)


# =========================================================
# 7. CREATE ONE REMAINING ROW PER ORDER_ID
# =========================================================
for _, row in order_level_rows.iterrows():
    remaining_quantity = pd.to_numeric(
        row["remaining_quantity"],
        errors="coerce",
    )

    if (
        pd.isna(remaining_quantity)
        or remaining_quantity <= 0
    ):
        continue

    remaining_colour = (
        get_remaining_colour_code(
            original_colour=(
                row["colour_code"]
            ),
            total_allocated_quantity=(
                row[
                    "total_allocated_quantity"
                ]
            ),
        )
    )

    if remaining_colour is None:
        continue

    sap_output_rows.append(
        {
            # Internal key. Removed from final SAP output.
            "order_id": row["order_id"],

            # Remaining quantity stays at the parent plant.
            "From Location": clean_code(
                row["parent_plant"]
            ),

            "Material Code": clean_code(
                row["material_code"]
            ),

            "To Location": clean_code(
                row["dealer_code"]
            ),

            "Quantity": format_sap_quantity(
                remaining_quantity
            ),

            # Colour logic:
            #
            # Original 16:
            #   no allocation -> 6
            #   partial       -> 7
            #
            # Original 17:
            #   no allocation -> 7
            #   partial       -> 8
            "Colour Code": remaining_colour,

            "Order Type": (
                row["order_category"]
            ),

            "DATE": row["DATE"],

            "TIME": row["TIME"],

            "GIT Flag": 0,

            # Remaining row appears last.
            "_row_type_order": 3,
        }
    )


# =========================================================
# 8. CREATE FINAL SAP OUTPUT
# =========================================================
sap_output_with_keys = pd.DataFrame(
    sap_output_rows
)

final_columns = [
    "From Location",
    "Material Code",
    "To Location",
    "Quantity",
    "Colour Code",
    "Order Type",
    "DATE",
    "TIME",
    "GIT Flag",
]


if sap_output_with_keys.empty:
    sap_output = pd.DataFrame(
        columns=final_columns
    )

else:
    # Physical rows first, GIT rows second,
    # remaining quantity rows last.
    sap_output_with_keys = (
        sap_output_with_keys.sort_values(
            by=[
                "order_id",
                "_row_type_order",
                "From Location",
            ],
            ascending=[
                True,
                True,
                True,
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    sap_output = (
        sap_output_with_keys[
            final_columns
        ]
        .copy()
    )


# =========================================================
# 9. FINAL DATA-TYPE CLEANUP
# =========================================================
sap_output["Quantity"] = pd.to_numeric(
    sap_output["Quantity"],
    errors="coerce",
).fillna(0)


sap_output["Colour Code"] = pd.to_numeric(
    sap_output["Colour Code"],
    errors="coerce",
).astype("Int64")


sap_output["GIT Flag"] = pd.to_numeric(
    sap_output["GIT Flag"],
    errors="coerce",
).fillna(0).astype("int64")


# Keep DATE and TIME as strings so leading zeros are retained.
sap_output["DATE"] = (
    sap_output["DATE"]
    .astype("string")
)

sap_output["TIME"] = (
    sap_output["TIME"]
    .astype("string")
    .str.zfill(6)
)


# =========================================================
# 10. VALIDATE FINAL SAP OUTPUT
# =========================================================
invalid_colour_rows = sap_output[
    sap_output["Colour Code"].isna()
]

if not invalid_colour_rows.empty:
    raise ValueError(
        "SAP output contains invalid colour codes.\n"
        f"{invalid_colour_rows.head(20).to_string(index=False)}"
    )


invalid_quantity_rows = sap_output[
    sap_output["Quantity"] <= 0
]

if not invalid_quantity_rows.empty:
    raise ValueError(
        "SAP output contains zero or negative quantities.\n"
        f"{invalid_quantity_rows.head(20).to_string(index=False)}"
    )


invalid_from_location_rows = sap_output[
    sap_output["From Location"].isna()
    | sap_output[
        "From Location"
    ].astype("string").str.strip().eq("")
]

if not invalid_from_location_rows.empty:
    raise ValueError(
        "SAP output contains blank From Location values.\n"
        f"{invalid_from_location_rows.head(20).to_string(index=False)}"
    )


# =========================================================
# 11. OPTIONAL RECONCILIATION VALIDATION
# =========================================================
# For every order:
#
# physical allocation
# + GIT reservation
# + remaining quantity
# should equal original requested quantity.
# =========================================================
if (
    "original_requested_quantity"
    in sap_df.columns
):
    sap_df[
        "original_requested_quantity"
    ] = pd.to_numeric(
        sap_df[
            "original_requested_quantity"
        ],
        errors="coerce",
    ).fillna(0)

    requested_quantity_by_order = (
        sap_df.groupby(
            "order_id",
            as_index=False,
            dropna=False,
        )
        .agg(
            original_requested_quantity=(
                "original_requested_quantity",
                "max",
            )
        )
    )

    reconciliation_df = (
        order_level_rows.merge(
            requested_quantity_by_order,
            on="order_id",
            how="left",
            validate="one_to_one",
        )
    )

    reconciliation_df[
        "reconciled_quantity"
    ] = (
        reconciliation_df[
            "physical_allocated_quantity"
        ]
        + reconciliation_df[
            "git_reserved_quantity"
        ]
        + reconciliation_df[
            "remaining_quantity"
        ]
    )

    reconciliation_df[
        "quantity_difference"
    ] = (
        reconciliation_df[
            "original_requested_quantity"
        ]
        - reconciliation_df[
            "reconciled_quantity"
        ]
    )

    invalid_reconciliation = (
        reconciliation_df[
            reconciliation_df[
                "quantity_difference"
            ].abs() > 0.000001
        ]
    )

    print(
        "Orders failing quantity reconciliation:",
        len(invalid_reconciliation),
    )

    if not invalid_reconciliation.empty:
        display(
            invalid_reconciliation.head(20)
        )


# =========================================================
# 12. PREVIEW FINAL OUTPUT
# =========================================================
print()
print(
    "Input decision rows:",
    len(sap_df),
)

print(
    "Unique order IDs:",
    sap_df["order_id"].nunique(),
)

print(
    "SAP output rows:",
    len(sap_output),
)

print()
print("SAP output colour distribution:")

print(
    sap_output[
        "Colour Code"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

print()
display(
    sap_output.head(20)
)

Orders failing quantity reconciliation: 0

Input decision rows: 2603
Unique order IDs: 2596
SAP output rows: 2610

SAP output colour distribution:
Colour Code
1     643
2      77
6    1646
7     240
8       4
Name: count, dtype: Int64



,From Location,Material Code,To Location,Quantity,Colour Code,Order Type,DATE,TIME,GIT Flag
0,1155,IB002785,1S1180,45,6,ZFMO,20261030,030107,0
1,1157,IB002785,1S6189,21,6,ZFMO,20261030,030107,0
2,1153,2221528644,1S6050,3,6,ZFMO,20261030,030107,0
3,1155,2223795158,1S1180,1,6,ZFMO,20261030,030107,0
4,1159,IA341104,1S3466,1,6,ZMSO,20261030,030107,0
5,1159,IE450249,1S5107,1,6,ZMSO,20261030,030107,0
6,1110,IE450249,1S3467,1,6,ZMSO,20261030,030107,0
7,1110,IE450249,1S3425,1,6,ZMSO,20261030,030107,0
8,1161,IE468534,1S1120,1,1,ZMSO,20261030,030107,0
9,1153,IM301658,1S6098,1,6,ZMSO,20261030,030107,0


In [16]:
ration_qty = pd.read_csv(r"C:\Order Orchestration code\order_orchestration\Regular_original_files\30_10_2026\YVF_RATION_QTY.csv")

In [17]:
ration_qty.head()

,From Location,Material Code,To Location,Quantity,Colour Code,Order Type,DATE,TIME
0,1110,IE451619,1S3486,2,1,ZMSO,20261030,30107
1,1155,IE452257,1S1218,1,9,ZMSO,20261030,30107
2,1157,IE452328,1S6112,1,1,ZMSO,20261030,30107
3,1110,IE452328,1S7084,1,3,ZMSO,20261030,30107
4,1158,IE452328,1S7174,2,3,ZMSO,20261030,30107


In [18]:
sap_output.head()

,From Location,Material Code,To Location,Quantity,Colour Code,Order Type,DATE,TIME,GIT Flag
0,1155,IB002785,1S1180,45,6,ZFMO,20261030,030107,0
1,1157,IB002785,1S6189,21,6,ZFMO,20261030,030107,0
2,1153,2221528644,1S6050,3,6,ZFMO,20261030,030107,0
3,1155,2223795158,1S1180,1,6,ZFMO,20261030,030107,0
4,1159,IA341104,1S3466,1,6,ZMSO,20261030,030107,0


In [19]:
import pandas as pd


# ---------------------------------------------------------
# 1. Create working copies
# ---------------------------------------------------------
generated_sap_df = sap_output.copy()
original_sap_df = ration_qty.copy()


# ---------------------------------------------------------
# 2. Standardize column names
# ---------------------------------------------------------
generated_sap_df.columns = (
    generated_sap_df.columns
    .astype(str)
    .str.strip()
)

original_sap_df.columns = (
    original_sap_df.columns
    .astype(str)
    .str.strip()
)


# ---------------------------------------------------------
# 3. Validate required original SAP columns
# ---------------------------------------------------------
required_sap_columns = [
    "From Location",
    "Material Code",
    "To Location",
    "Quantity",
    "Colour Code",
    "Order Type",
    "DATE",
    "TIME",
]

missing_original_columns = [
    column
    for column in required_sap_columns
    if column not in original_sap_df.columns
]

if missing_original_columns:
    raise ValueError(
        "Original SAP dataframe is missing columns: "
        f"{missing_original_columns}. "
        f"Available columns: "
        f"{original_sap_df.columns.tolist()}"
    )


# ---------------------------------------------------------
# 4. Convert original colour code for filtering
# ---------------------------------------------------------
original_colour_numeric = pd.to_numeric(
    original_sap_df["Colour Code"],
    errors="coerce",
)


# ---------------------------------------------------------
# 5. Select original colour 8, 9, and 10 rows
# ---------------------------------------------------------
original_8_9_10_df = (
    original_sap_df.loc[
        original_colour_numeric.isin(
            [8, 9, 10]
        ),
        required_sap_columns,
    ]
    .copy()
)


# ---------------------------------------------------------
# 6. Add GIT Flag to original rows
# ---------------------------------------------------------
original_8_9_10_df["GIT Flag"] = 0


# ---------------------------------------------------------
# 7. Match the generated SAP column order
# ---------------------------------------------------------
final_sap_columns = [
    "From Location",
    "Material Code",
    "To Location",
    "Quantity",
    "Colour Code",
    "Order Type",
    "DATE",
    "TIME",
    "GIT Flag",
]

# Validate generated SAP columns.
missing_generated_columns = [
    column
    for column in final_sap_columns
    if column not in generated_sap_df.columns
]

if missing_generated_columns:
    raise ValueError(
        "Generated SAP dataframe is missing columns: "
        f"{missing_generated_columns}. "
        f"Available columns: "
        f"{generated_sap_df.columns.tolist()}"
    )

generated_sap_df = generated_sap_df[
    final_sap_columns
].copy()

original_8_9_10_df = original_8_9_10_df[
    final_sap_columns
].copy()


# ---------------------------------------------------------
# 8. Normalize important data types
# ---------------------------------------------------------
for current_df in [
    generated_sap_df,
    original_8_9_10_df,
]:
    current_df["Colour Code"] = pd.to_numeric(
        current_df["Colour Code"],
        errors="coerce",
    ).astype("Int64")

    current_df["Quantity"] = pd.to_numeric(
        current_df["Quantity"],
        errors="coerce",
    )

    current_df["GIT Flag"] = pd.to_numeric(
        current_df["GIT Flag"],
        errors="coerce",
    ).fillna(0).astype("int64")

    # Preserve leading zeros in date and time.
    current_df["DATE"] = (
        current_df["DATE"]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .str.zfill(8)
    )

    current_df["TIME"] = (
        current_df["TIME"]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .str.zfill(6)
    )

    # Clean location and code columns.
    for column in [
        "From Location",
        "Material Code",
        "To Location",
        "Order Type",
    ]:
        current_df[column] = (
            current_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
            .str.replace(
                r"\.0$",
                "",
                regex=True,
            )
        )


# ---------------------------------------------------------
# 9. Append original 8, 9, and 10 rows
# ---------------------------------------------------------
final_sap_output = pd.concat(
    [
        generated_sap_df,
        original_8_9_10_df,
    ],
    ignore_index=True,
    sort=False,
)


# ---------------------------------------------------------
# 10. Keep final column order
# ---------------------------------------------------------
final_sap_output = final_sap_output[
    final_sap_columns
].copy()


# ---------------------------------------------------------
# 11. Print validation summary
# ---------------------------------------------------------
print(
    "Generated SAP rows:",
    len(generated_sap_df),
)

print(
    "Original colour 8, 9, 10 rows appended:",
    len(original_8_9_10_df),
)

print(
    "Final SAP output rows:",
    len(final_sap_output),
)

print()
print("Final colour-code distribution:")

print(
    final_sap_output["Colour Code"]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

print()
display(
    final_sap_output.head(20)
)

Generated SAP rows: 2610
Original colour 8, 9, 10 rows appended: 7194
Final SAP output rows: 9804

Final colour-code distribution:
Colour Code
1      643
2       77
6     1646
7      240
8     3078
9     4079
10      41
Name: count, dtype: Int64



,From Location,Material Code,To Location,Quantity,Colour Code,Order Type,DATE,TIME,GIT Flag
0,1155,IB002785,1S1180,45,6,ZFMO,20261030,030107,0
1,1157,IB002785,1S6189,21,6,ZFMO,20261030,030107,0
2,1153,2221528644,1S6050,3,6,ZFMO,20261030,030107,0
3,1155,2223795158,1S1180,1,6,ZFMO,20261030,030107,0
4,1159,IA341104,1S3466,1,6,ZMSO,20261030,030107,0
5,1159,IE450249,1S5107,1,6,ZMSO,20261030,030107,0
6,1110,IE450249,1S3467,1,6,ZMSO,20261030,030107,0
7,1110,IE450249,1S3425,1,6,ZMSO,20261030,030107,0
8,1161,IE468534,1S1120,1,1,ZMSO,20261030,030107,0
9,1153,IM301658,1S6098,1,6,ZMSO,20261030,030107,0


In [21]:
sap_output.to_csv(r"C:\Order Orchestration code\order_orchestration\data\outputs_30oct26\order_orchestration_30102026.csv", index=False)